# R02 · Cookbook-Referenzlabor: Adressen, die Zahl N, Datentypen, Bereiche

**NHyDB-0.27.1-Notebook-Reihe · v1 · Referenzteil**

Diese Beispiele basieren auf dem offiziellen
[NHyDB Python Cookbook](https://github.com/nhydb/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (Abschnitte 9–12), das als maßgebliche Referenz dient.
Code und erwartete Ausgaben sind unverändert aus dem Cookbook übernommen, die Begleittexte ins
Deutsche übersetzt; dieselben Beispiele führt auch die NHyDB-Testsuite aus.

* Jede Code-Zelle ist ein **eigenständiges Beispiel** – führe eine beliebige Zelle für sich aus oder nutze *Run All*.
* Jedes Beispiel legt seinen eigenen temporären Datenbankordner an.
* Unter jedem Beispiel findest du die im Cookbook abgedruckte Ausgabe zum Vergleich mit deiner.
* Die Hauptreihe (`../main/`) erklärt die Ideen hinter diesen APIs Schritt für Schritt.

**Inhalt** (11 ausführbare Beispiele)

* 9. Adressen und die Zahl N
* 10. Lesen über N und Adressen untersuchen
* 11. Datentypen
* 12. Viele Datensätze lesen: Bereiche und Iteratoren

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.27.1':
    print('⚠ These examples were written for NHyDB 0.27.1; results may differ.')

nhydb 0.27.1 · file format 0.16


## 9. Adressen und die Zahl N

Die Funktionen auf Modulebene funktionieren ohne Datenbank.

* `dynadron_n(realm, u, v, lane)` – Adresse → `N`.
* `n_to_dynadron_address(N)` – `N` (int oder Text `"N…"`) → Adresse.
* `cell_address(realm, u, v, lane)` / `dynadron_address(realm, u, v)` –
  Adresse mit allen abgeleiteten Kennzahlen.

In [2]:
import nhydb

n_left = nhydb.dynadron_n(1, 11, 13, "NL")
n_right = nhydb.dynadron_n(1, 11, 13, "NR")
print(n_left, n_right)

a = nhydb.n_to_dynadron_address(589)
print(a["realm"], a["u"], a["v_local"], a["lane_name"], a["G"])
print(nhydb.n_to_dynadron_address("N565")["lane_name"])
print(nhydb.parse_n_reference("N589"))

c = nhydb.cell_address(1, 11, 13, 1)
print(c["N_ref"], c["G"], c["NL"], c["NR"])

565 589
1 11 13 NR 24
NL
589
N589 24 565 589


*Erwartete Ausgabe (aus dem Cookbook):*

```text
565 589
1 11 13 NR 24
NL
589
N589 24 565 589
```

Für Realm 1 lautet die Formel `G = u + v`, `N = G² − u` (NL) oder `N = G² + v`
(NR); zum Beispiel `24² − 11 = 565`. Größere Realms verwenden die globale
Koordinate `v_global = (R − 1)·10^100 + v`, sodass `N` sehr groß wird –
Python-Ganzzahlen rechnen damit exakt:

In [3]:
import nhydb

n = nhydb.dynadron_n(23, 14, 744, "NL")
print(len(str(n)))                                   # number of digits
a = nhydb.n_to_dynadron_address(n)
print(a["realm"], a["u"], a["v_local"], a["lane_name"])

print(nhydb.REALM_SPAN == 10**100)
print(nhydb.global_v(2, 5) == 10**100 + 5, nhydb.split_global_v(10**100 + 5))
lo, hi = nhydb.realm_v_bounds(2)
print(lo == 10**100 + 1, hi == 2 * 10**100)
print(nhydb.validate_local_v(5))
print(nhydb.normalize_lane("nr"), nhydb.lane_name(0))
print(nhydb.dynadron_address(1, 11, 13)["G"])        # indicators without lane

203
23 14 744 NL
True
True (2, 5)
True True
5
1 NL
24


*Erwartete Ausgabe (aus dem Cookbook):*

```text
203
23 14 744 NL
True
True (2, 5)
True True
5
1 NL
24
```

Nicht jede natürliche Zahl ist eine Speicherzelle: Zahlen auf den Linien „GZ“ und „N0“
des Dynadrons sind ausgeschlossen und lösen `AddressError` aus:

In [4]:
import nhydb

for n in (4, 576, 589):
    try:
        print(n, nhydb.n_to_dynadron_address(n)["u"])
    except nhydb.AddressError as e:
        print(n, "AddressError:", e)

4 AddressError: N4 is a GZ point, not an NHyDB storage cell
576 AddressError: N576 is a GZ point, not an NHyDB storage cell
589 11


*Erwartete Ausgabe (aus dem Cookbook):*

```text
4 AddressError: N4 is a GZ point, not an NHyDB storage cell
576 AddressError: N576 is a GZ point, not an NHyDB storage cell
589 11
```

Koordinaten außerhalb des gültigen Bereichs werden ebenfalls abgelehnt (`v` muss innerhalb
seines Realms bleiben: `1 ≤ v ≤ 10^100`):

In [5]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "limits.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    try:
        db.put_cell(1, 14, 10**100 + 1, 5, type="u8")
    except nhydb.RealmLimitError as e:
        print("RealmLimitError")
    print(db.realm_bounds(1)["local_v_max"] == 10**100)

RealmLimitError
True


*Erwartete Ausgabe (aus dem Cookbook):*

```text
RealmLimitError
True
```

Die Low-Level-Codecs `enc_vu`/`dec_vu` (VarUInt) und `enc_vi`/`dec_vi`
(ZigZag-VarInt) werden ebenfalls exportiert:

In [6]:
import nhydb

data = nhydb.enc_vu(300)
print(data, nhydb.dec_vu(data))        # (value, bytes consumed)
print(nhydb.dec_vi(nhydb.enc_vi(-3)))

b'\xac\x02' (300, 2)
(-3, 1)


*Erwartete Ausgabe (aus dem Cookbook):*

```text
b'\xac\x02' (300, 2)
(-3, 1)
```

## 10. Lesen über N und Adressen untersuchen

Eine Zahl `N` benennt genau eine Zelle. Wie gelesen wird, hängt vom Modus ab:

| Modus | Ergebnis |
|---|---|
| `"cell"` | genau die adressierte Zelle |
| `"record"` | der ganze logische Datensatz, zu dem die Zelle gehört (über ihr Layout) |
| `"auto"` (Standard) | Datensatz, wenn `N` die Ankerzelle des Datensatzes ist, sonst Zelle |

In [7]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "byn.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "shop")
    db.create_shape(1, 1, "order", [
        {"v": 1, "name": "id", "type": "varuint"},
        {"v": 2, "name": "customer", "type": "utf8"},
        {"v": 3, "name": "amount", "type": "f64"},
    ])
    db.create_classic_layout(1, 1, "orders", shape_id=1)
    db.put_dense_batch(1, 1, [[1, "Alice", 19.9], [2, "Bob", 5.5]], commit=True)

    n = nhydb.dynadron_n(1, 2, 3, "NL")                 # amount of order 2
    cell = db.get_n(n, mode="cell")
    print(cell["N"], cell["field"], cell["value"], cell["record_N"])

    rec = db.get_n(cell["record_N"])                    # anchor -> whole record
    print(rec["effective_mode"] if "effective_mode" in rec else rec["result"], rec["data"])

    print(db.get_address(1, 2, 3, mode="cell")["value"])
    print(db.get_n_cell(n)["value"], db.get_n_record(n)["object"])
    print(db.get_cell_n(n, dyn_address=False)["value"])

N23 amount 5.5 N7
record {'id': 2, 'customer': 'Bob', 'amount': 5.5}
5.5
5.5 2
5.5


*Erwartete Ausgabe (aus dem Cookbook):*

```text
N23 amount 5.5 N7
record {'id': 2, 'customer': 'Bob', 'amount': 5.5}
5.5
5.5 2
5.5
```

Weitere Werkzeuge, um eine Adresse oder einen Datensatz zu verstehen:

In [8]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "inspect.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "shop")
    db.create_shape(1, 1, "order", [
        {"v": 1, "name": "id", "type": "varuint"},
        {"v": 2, "name": "customer", "type": "utf8"},
    ])
    db.create_classic_layout(1, 1, "orders", shape_id=1)
    db.put_dense_batch(1, 1, [[1, "Alice"], [2, "Bob"]], commit=True)

    d = db.describe_n(14)                               # N14 = realm 1, u=2, v=2, NL
    print(d["u"], d["v"], d["lane_name"], d["layout"]["name"])
    print(db.resolve_n("N14")["realm_defined"])
    print(db.record_n_for_address(1, 2, 2), db.record_n_from_n(14))
    print(db.record_anchor(1, 2)["N_ref"], db.logical_address(1, 2, "customer")["N_ref"])
    print(db.occupied_n(1, 2))
    print(db.occupied_from_n(14, output="N")["cells"])
    view = db.get_record_view(1, 2)
    print(view["cell_count"], [c["N"] for c in view["cells"]])
    print(db.cell_address(1, 2, 2)["N"], db.dyn_address(1, 2, 2)["NR"])

2 2 NL orders
True
N7 N7
N7 N14
['N7', 'N14']
['N7', 'N14']
2 ['N7', 'N14']
14 18


*Erwartete Ausgabe (aus dem Cookbook):*

```text
2 2 NL orders
True
N7 N7
N7 N14
['N7', 'N14']
['N7', 'N14']
2 ['N7', 'N14']
14 18
```

## 11. Datentypen

| Typ | Python-Wert | Bereich / Hinweis |
|---|---|---|
| `u8`, `u16`, `u32`, `u64` | `int` | vorzeichenlos, geprüft |
| `i8`, `i16`, `i32`, `i64` | `int` | vorzeichenbehaftet, geprüft |
| `varuint` | `int` ≥ 0 | beliebig groß |
| `varint` | `int` | beliebig groß, negativ erlaubt |
| `f32`, `f64` | `float` | `f32`: Werte, die überlaufen würden, werden abgelehnt |
| `bool` | `bool` | |
| `utf8` | `str` | |
| `bytes` | `bytes` | |

`nhydb.TYPE_CODES` bildet die Namen auf die in der Datei gespeicherten Codes ab.

In [9]:
import os
import tempfile

import nhydb

print(nhydb.TYPE_CODES["utf8"], len(nhydb.TYPE_CODES))
fields = [("u8", 255), ("u16", 65535), ("u32", 4_000_000_000), ("u64", 2**64 - 1),
          ("i8", -128), ("i16", -30000), ("i32", -2**31), ("i64", -2**63),
          ("f32", 1.5), ("f64", 3.141592653589793), ("bool", True),
          ("varuint", 10**40), ("varint", -10**40), ("utf8", "Grüße ✓"), ("bytes", b"\x00\xff")]

path = os.path.join(tempfile.mkdtemp(), "types.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "types")
    db.create_shape(1, 1, "all", [{"v": i + 1, "name": t, "type": t} for i, (t, _) in enumerate(fields)])
    db.create_classic_layout(1, 1, "all", shape_id=1)
    db.put_record(1, 1, {t: value for t, value in fields}, commit=True)
    rec = db.get_record(1, 1)
    print(all(rec[t] == value for t, value in fields))
    try:
        db.put_record(1, 2, {**{t: value for t, value in fields}, "u8": 256})
    except ValueError:
        print("u8 out of range -> ValueError")

14 15
True
u8 out of range -> ValueError


*Erwartete Ausgabe (aus dem Cookbook):*

```text
14 15
True
u8 out of range -> ValueError
```

## 12. Viele Datensätze lesen: Bereiche und Iteratoren

`range`, `select`, `select_us`, `find` (Abschnitt 13) und die
`iter_materialized_*`-Methoden liefern **verzögerte Iteratoren** (lazy): Datensätze werden
Block für Block geladen, der Speicherbedarf hängt also nicht von der Größe des Ergebnisses ab.
Lücken in `u` werden effizient übersprungen.

In [10]:
import itertools
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "iter.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "log")
    db.create_shape(1, 1, "entry", [{"v": 1, "name": "level", "type": "utf8"},
                                    {"v": 2, "name": "code", "type": "u16"}])
    db.create_classic_layout(1, 1, "log", shape_id=1)
    db.put_dense_batch(1, 1, [["info" if u % 3 else "warn", u] for u in range(1, 1001)])
    db.put(1, 10**15, ["error", 999])                   # far away: the gap is skipped
    db.commit()

    first = list(itertools.islice(db.range(1, 1, 10**15), 2))
    print([r["code"] for r in first])
    print(sum(1 for _ in db.range(1, 1, 10**15)))
    print(list(db.select(1, 998, 1000, ["code"])))
    print([r["level"] for r in db.select_us(1, [3, 10**15], ["level"])])

[1, 2]
1001
[{'_u': 998, '_lane': 0, '_lane_name': 'NL', 'code': 998}, {'_u': 999, '_lane': 0, '_lane_name': 'NL', 'code': 999}, {'_u': 1000, '_lane': 0, '_lane_name': 'NL', 'code': 1000}]
['warn', 'error']


*Erwartete Ausgabe (aus dem Cookbook):*

```text
[1, 2]
1001
[{'_u': 998, '_lane': 0, '_lane_name': 'NL', 'code': 998}, {'_u': 999, '_lane': 0, '_lane_name': 'NL', 'code': 999}, {'_u': 1000, '_lane': 0, '_lane_name': 'NL', 'code': 1000}]
['warn', 'error']
```

So listest du alles auf, was physisch gespeichert ist (dense und sparse):

In [11]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "walk.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u8"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.put_dense_batch(1, 1, [[10], [20]])
    db.put_cell(1, 5, 9, "loose", type="utf8")
    db.commit()

    for cell in db.iter_materialized_cells(1):
        print(cell["N"], cell["u"], cell["v"], cell["value"])
    print(list(db.iter_materialized_sparse_points(1)))
    print(list(db.iter_materialized_record_ns(1)))

N3 1 1 10
N7 2 1 20
N191 5 9 loose
[{'realm': 1, 'lane': 0, 'u': 5, 'v': 9, 'value': 'loose', 'type': 'utf8'}]
['N3', 'N7', 'N31']


*Erwartete Ausgabe (aus dem Cookbook):*

```text
N3 1 1 10
N7 2 1 20
N191 5 9 loose
[{'realm': 1, 'lane': 0, 'u': 5, 'v': 9, 'value': 'loose', 'type': 'utf8'}]
['N3', 'N7', 'N31']
```

**Adresskennzahlen in Ergebnissen.** `get`, `range` und `find` können die
mathematischen Kennzahlen eines Feldes (`dyn_address="once"` mit `dyn_v`) oder
aller Felder (`dyn_address="auto"`) hinzufügen. `set_dyn_address_output(mode, v=…)`
legt einen Standard für das Datenbankobjekt fest (`"off"`, `"once"`, `"auto"`).

In [12]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "dyn.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "a", "type": "u8"}, {"v": 2, "name": "b", "type": "u8"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.put(1, 1, [7, 8], commit=True)

    r = db.get(1, 1, dyn_address="once", dyn_v=2)
    print(r["_dyn_v"], r["_dyn_NL"], r["_dyn_NR"])
    r = db.get(1, 1, dyn_address="auto")
    print(sorted(r["_dyn_addresses"]), r["_dyn_addresses"]["a"]["NL"])
    db.set_dyn_address_output("once", v=1)
    print(db.get(1, 1)["_dyn_NL"])
    db.set_dyn_address_output("off")
    print("_dyn_NL" in db.get(1, 1))

2 8 11
['a', 'b'] 3
3
False


*Erwartete Ausgabe (aus dem Cookbook):*

```text
2 8 11
['a', 'b'] 3
3
False
```